In [4]:
using Pkg; Pkg.activate(".")
using QuantumCircuitsMPS
using ITensors, ITensorMPS
using LinearAlgebra
using Random

  Activating project at `~/personal/QuantumCircuitsMPS.jl`


In [5]:
# Extract state vector from any L-qubit MPS (s1-fastest column-major basis)
function mps_to_statevector(state)
    T = state.mps[1]
    for i in 2:state.L
        T *= state.mps[i]
    end
    return vec(array(T, state.sites...))
end

# Apply a custom k-qubit gate matrix U (2^k × 2^k) to physical sites phy_sites.
# Uses the framework's apply_op_internal! with ITensor built from U.
# Convention: ITensor(reshape(U, d,...,d), s1,...,sk, s1',...,sk')
# The effective operation is U^T * ψ (matching how HaarRandom works).
function apply_gate!(state, U::AbstractMatrix, phy_sites::Vector{Int})
    k = length(phy_sites)
    d = state.local_dim
    ram_sites = [state.phy_ram[ps] for ps in phy_sites]
    site_indices = [state.sites[rs] for rs in ram_sites]
    dims = ntuple(_ -> d, 2k)
    U_tensor = reshape(U, dims...)
    all_inds = vcat(site_indices, prime.(site_indices))
    op = ITensor(U_tensor, all_inds...)
    apply_op_internal!(state.mps, op, state.sites, state.cutoff, state.maxdim)
end

# Apply a k-qubit gate matrix U to state vector ψ (length 2^L) on phy_sites.
# Matches MPS convention: effective operation is U^T on target site subspace.
function sv_apply(ψ::AbstractVector, U::AbstractMatrix, phy_sites::Vector{Int}, L::Int)
    d = 2
    k = length(phy_sites)
    sorted_sites = sort(phy_sites)
    other_sites = setdiff(1:L, sorted_sites)
    perm = vcat(sorted_sites, other_sites)
    iperm = invperm(perm)
    ψ_arr = reshape(copy(ψ), ntuple(_ -> d, L)...)
    ψ_perm = permutedims(ψ_arr, perm)
    shape = size(ψ_perm)
    ψ_mat = reshape(ψ_perm, d^k, d^length(other_sites))
    ψ_mat_out = transpose(U) * ψ_mat
    ψ_out = permutedims(reshape(ψ_mat_out, shape), iperm)
    return vec(ψ_out)
end

# Project site to |outcome⟩ in a state vector, then normalize.
function sv_project(ψ::AbstractVector, site::Int, outcome::Int, L::Int)
    d = 2
    ψ_arr = reshape(copy(ψ), ntuple(_ -> d, L)...)
    for idx in CartesianIndices(ψ_arr)
        if idx[site] != outcome + 1
            ψ_arr[idx] = 0
        end
    end
    ψ_out = vec(ψ_arr)
    return ψ_out / norm(ψ_out)
end

# Generate a Haar random unitary (exact CT.jl algorithm)
function haar_random_unitary(seed::Int; dim=4)
    rng = MersenneTwister(seed)
    z = randn(rng, dim, dim) + randn(rng, dim, dim) * im
    Q, R = qr(z); Q = Matrix(Q)
    return Q * Diagonal(diag(R) ./ abs.(diag(R)))
end

println("Helpers loaded: mps_to_statevector, apply_gate!, sv_apply, sv_project, haar_random_unitary")

Helpers loaded: mps_to_statevector, apply_gate!, sv_apply, sv_project, haar_random_unitary


In [6]:
# --- Test 1: L=2, HaarRandom unitary ---
U = haar_random_unitary(42)
rng1 = RNGRegistry(gates_spacetime=1, gates_realization=42, born_measurement=2)
s1 = SimulationState(L=2, bc=:open, maxdim=64, rng=rng1)
initialize!(s1, ProductState(binary_int=0))
ψ0 = mps_to_statevector(s1)
apply_gate!(s1, U, [1, 2])
ψ_mps = mps_to_statevector(s1)
ψ_sv = sv_apply(ψ0, U, [1, 2], 2)
println("Test 1 (L=2, gate [1,2]): norm = ", norm(ψ_mps - ψ_sv))

# --- Test 2: L=2, Projection ---
rng2 = RNGRegistry(gates_spacetime=1, gates_realization=42, born_measurement=2)
s2 = SimulationState(L=2, bc=:open, maxdim=64, rng=rng2)
initialize!(s2, ProductState(binary_int=0))
apply_gate!(s2, U, [1, 2])
ψ_pre = mps_to_statevector(s2)
apply!(s2, Projection(0), SingleSite(1))
ψ_mps2 = mps_to_statevector(s2)
ψ_sv2 = sv_project(ψ_pre, 1, 0, 2)
println("Test 2 (L=2, proj site 1): norm = ", norm(ψ_mps2 - ψ_sv2))

# --- Test 3: L=4, gate on sites [2,3] ---
U3 = haar_random_unitary(77)
rng3 = RNGRegistry(gates_spacetime=1, gates_realization=99, born_measurement=2)
s3 = SimulationState(L=4, bc=:open, maxdim=64, rng=rng3)
initialize!(s3, ProductState(binary_int=5))
ψ0_3 = mps_to_statevector(s3)
apply_gate!(s3, U3, [2, 3])
ψ_mps3 = mps_to_statevector(s3)
ψ_sv3 = sv_apply(ψ0_3, U3, [2, 3], 4)
println("Test 3 (L=4, gate [2,3]): norm = ", norm(ψ_mps3 - ψ_sv3))

# --- Test 4: L=3, two sequential gates ---
Ua = haar_random_unitary(11)
Ub = haar_random_unitary(22)
rng5 = RNGRegistry(gates_spacetime=1, gates_realization=1, born_measurement=2)
s5 = SimulationState(L=3, bc=:open, maxdim=64, rng=rng5)
initialize!(s5, ProductState(binary_int=0))
ψ0_5 = mps_to_statevector(s5)
apply_gate!(s5, Ua, [1, 2])
apply_gate!(s5, Ub, [2, 3])
ψ_mps5 = mps_to_statevector(s5)
ψ_sv5 = sv_apply(sv_apply(ψ0_5, Ua, [1, 2], 3), Ub, [2, 3], 3)
println("Test 4 (L=3, two gates): norm = ", norm(ψ_mps5 - ψ_sv5))

Test 1 (L=2, gate [1,2]): norm = 5.329075035707888e-16
Test 2 (L=2, proj site 1): norm = 2.618455766672135e-16
Test 3 (L=4, gate [2,3]): norm = 1.368563798628543e-16
Test 4 (L=3, two gates): norm = 5.275499161464325e-16
